# KhaledDSML 15 — Research Monitoring & Meta-analysis
## مراقبة انجراف البيانات والتحليل البعدي للدراسات

**v0.10.0 stable candidate source / مصدر الإصدار النهائي قبل النشر.** Stable published release remains 0.9.0. Synthetic examples only. All results are computed from the inputs. / الأمثلة اصطناعية وليست نتائج دراسات حقيقية.
**Source for development testing: 0.10.0 (release candidate), NOT published stable PyPI; official previous stable 0.9.0. / نسخة مصدرية تجريبية وليست إصدارًا مستقرًا منشورًا. 


In [ ]:
%pip install -q "git+https://github.com/noble2013/KhaledDSML.git@main"

## 1 — Quantile-based drift / فحص انجراف البيانات

PSI وJensen–Shannon وWasserstein مؤشرات وصفية، وليست قيمة p. اختبار KS افتراضي ويعتمد على استقلالية العيّنات. / PSI, JS, and Wasserstein are descriptive; KS p-values assume independent samples.

In [ ]:
import numpy as np
from khaled_dsml import (
    distribution_drift_profile, welford_streaming_summary, sample_ratio_mismatch,
    meta_analysis_heterogeneity, meta_analysis_random_effects,
    egger_small_study_effects_test, display_analysis_dashboard,
)
rng = np.random.default_rng(2206)
reference = rng.normal(110, 19, 1000)
current = rng.normal(119, 19, 920)
drift = distribution_drift_profile(reference, current, bins=10)
display_analysis_dashboard(drift, theme="ocean")

## 2 — Welford streaming / تحديث المتوسط والتباين

تُقرأ الأجزاء تباعًا دون جمع السجلات في قائمة كبيرة؛ النتيجة وصفية فقط. / Stream chunks and obtain sample variance (ddof=1) without a complete in-memory array.

In [ ]:
def batches(values, size=180):
    for i in range(0, len(values), size):
        yield values[i:i+size]
running = welford_streaming_summary(batches(reference))
display_analysis_dashboard(running, theme="aurora")

## 3 — SRM allocation ratio / فحص انحراف نسب التوزيع

لا تستخدم نفس قيمة p لفحوص متكررة دون تصحيح المراقبة المتتابعة. / Intended for a single preregistered check, not repeated uncorrected peeking.

In [ ]:
allocation = sample_ratio_mismatch([565, 435], [.5, .5])
display_analysis_dashboard(allocation, theme="amethyst")

## 4 — Meta-analysis / التحليل البعدي

التأثيرات مستقلة وبالمقياس نفسه؛ الفواصل بطريقة DL تستعمل تقريب z وليس Hartung–Knapp. / Independent studies with comparable effects; DL normal intervals may undercover with few studies.

In [ ]:
effects = [.04, .27, .42, .14, -.09, .63, .32, .19, .47, -.13, .35, .18]
ses = [.09, .12, .10, .08, .11, .09, .14, .13, .10, .12, .09, .11]
display_analysis_dashboard(
    meta_analysis_heterogeneity(effects, ses), theme="ocean"
)
display_analysis_dashboard(
    meta_analysis_random_effects(effects, ses), theme="amethyst"
)

## 5 — Egger regression / انحدار إيغر

الانحياز المرتبط بحجم الدراسات ليس دليلًا قاطعًا على تحيز النشر، خصوصًا عند عدم التجانس. / Small-study effects are not proof of publication bias.

In [ ]:
egger = egger_small_study_effects_test(effects, ses)
display_analysis_dashboard(egger, theme="aurora")

## Research integrity / نزاهة البحث

- لا تستخدم PSI كاختبار دلالة.
- لا تتجاهل اعتماد القياسات أو الفحوص المتكررة.
- يجب ألا يكون تحليل Egger الوحيد للحكم على تحيز النشر.

**English:** No universal PSI cutoff, no causality, no uncorrected repeated monitoring and no substitution for specialist meta-analysis methodology.